# Notebook T4.3 — Feature Engineering Semántico con LLM
## Patrón: "El LLM como generador de features para ML"

**Referencia en la guía metodológica:** PASO 5 → PATRÓN C  
**Nivel:** Intermedio

---

### ¿Qué problema resuelve este notebook?

En muchos proyectos de ML tienes datos **mixtos**: algunas columnas son numéricas (precio, días de envío)
y otras son texto libre (reseñas de clientes, comentarios, descripciones).

El ML clásico ignora el texto porque no sabe qué hacer con él. Podrías usar TF-IDF o embeddings,
pero el LLM ofrece algo más potente: **features semánticas con significado de negocio**.

En lugar de "frecuencia de la palabra 'malo'", el LLM extrae "intención de no volver a comprar" —
una feature que un experto de negocio entendería y validaría.

**Caso de uso:** Predicción de recompra en e-commerce.  
El dataset tiene precio, días de envío **y** una reseña de texto libre.  
Comparamos un modelo que ignora el texto vs uno que usa features semánticas del LLM.

---

### Flujo del pipeline

```
features numéricas (precio, días envío)  ──────────────────────────┐
                                                                    ↓
texto de la reseña → LLM → features semánticas → concat → Modelo ML mejorado
                            (score_sentimiento,             ↓
                             intencion_recompra,       comparar con
                             menciona_devolucion...)   modelo baseline
```

---

### Conexión con la guía metodológica

- **Paso 1:** Datos mixtos (tabulares + texto); salida = clasificación de recompra  
- **Paso 4:** El LLM genera features numéricas a partir de texto (TIPO B de datos)  
- **Paso 5:** PATRÓN C — concatenación de features clásicas + semánticas  
- **Paso 6:** Comparación con modelo baseline para medir el valor del LLM

## PASO 2 — Instalación y Configuración

In [ ]:
!pip install langchain langchain-google-genai scikit-learn pandas numpy python-dotenv -q

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0  # extracción determinista
)
print("✅ Configuración lista")

## PASO 4 — Dataset de Reseñas de E-commerce

### Estructura del dataset: datos mixtos

Cada fila tiene:
- **Features numéricas**: `precio` y `dias_envio` — el ML clásico las usa directamente
- **Texto libre**: `resena` — el ML clásico la ignora; el LLM la transformará
- **Target**: `recompra` — ¿volvería a comprar el cliente? (1=sí, 0=no)

### ¿Por qué GradientBoosting en lugar de RandomForest?

`GradientBoostingClassifier` construye árboles de forma secuencial, cada uno corrigiendo 
los errores del anterior. Suele superar a RandomForest en datasets pequeños con buenas features.
Es la base de XGBoost y LightGBM, los algoritmos ganadores de competiciones de ML tabular.

In [ ]:
datos = [
    {"resena": "Excelente producto, superó mis expectativas. Lo recomiendo a todos mis amigos!", "precio": 45, "dias_envio": 2, "recompra": 1},
    {"resena": "Malísimo, se rompió a la semana. Una basura. Pido devolución urgente.", "precio": 30, "dias_envio": 5, "recompra": 0},
    {"resena": "El producto llegó bien, aunque el empaque estaba un poco dañado. Funciona ok.", "precio": 60, "dias_envio": 3, "recompra": 1},
    {"resena": "No entiendo cómo usar esto, las instrucciones son malísimas. Muy frustrado.", "precio": 25, "dias_envio": 1, "recompra": 0},
    {"resena": "Perfecto para regalo. Mi hijo quedó encantado. Definitivamente volvería a comprar.", "precio": 80, "dias_envio": 2, "recompra": 1},
    {"resena": "Tardó 15 días en llegar cuando decía 3. El producto en sí está bien pero el servicio horrible.", "precio": 55, "dias_envio": 15, "recompra": 0},
    {"resena": "Calidad aceptable para el precio. No es lo mejor del mercado pero cumple su función.", "precio": 20, "dias_envio": 4, "recompra": 1},
    {"resena": "Increíble calidad, materiales premium. Vale cada centavo.", "precio": 120, "dias_envio": 2, "recompra": 1},
    {"resena": "Color completamente diferente al de la foto. Me siento estafado.", "precio": 35, "dias_envio": 3, "recompra": 0},
    {"resena": "Buena relación calidad precio. El envío fue rápido y bien embalado.", "precio": 40, "dias_envio": 2, "recompra": 1},
    {"resena": "Segunda vez que compro este producto. Muy confiable y duradero.", "precio": 65, "dias_envio": 3, "recompra": 1},
    {"resena": "Esperaba más. Para ese precio hay mejores opciones en el mercado.", "precio": 90, "dias_envio": 4, "recompra": 0},
    {"resena": "Me llegó roto. El vendedor no responde. Eviten esta tienda.", "precio": 50, "dias_envio": 7, "recompra": 0},
    {"resena": "Funciona perfecto. Simple, efectivo y fácil de usar. 5 estrellas sin dudarlo.", "precio": 30, "dias_envio": 2, "recompra": 1},
    {"resena": "Regular. Ni muy bien ni muy mal. El envío fue lento pero llegó completo.", "precio": 45, "dias_envio": 8, "recompra": 0},
]

df = pd.DataFrame(datos)
print(f"Dataset: {len(df)} reseñas")
print(f"Tasa de recompra: {df['recompra'].mean():.1%}")
print(f"\nEjemplo de reseña positiva: '{datos[0]['resena']}'")
print(f"Ejemplo de reseña negativa: '{datos[1]['resena']}'")
df.head()

## PASO 5 (parte A) — Modelo Baseline: Solo Features Numéricas

### ¿Por qué empezar con un baseline?

El **baseline** es la referencia contra la que mides el impacto de añadir el LLM.  
Sin baseline, no puedes saber si el LLM realmente aporta valor o si el costo de la API 
no está justificado.

Siempre sigue este orden:
1. Entrena el modelo más simple posible (baseline)
2. Mide sus métricas
3. Añade complejidad (LLM features)
4. Mide de nuevo y compara

### Cross-validation con 3 folds

Con solo 15 muestras, el train/test split clásico sería muy inestable.  
`cross_val_score` divide los datos en 3 partes, entrena 3 veces (cada vez con una parte diferente 
como test) y promedia las métricas. Más robusto con datasets pequeños.

In [ ]:
features_base = ["precio", "dias_envio"]
X_base = df[features_base]
y = df["recompra"]

X_train, X_test, y_train, y_test = train_test_split(X_base, y, test_size=0.3, random_state=42)

modelo_base = GradientBoostingClassifier(n_estimators=50, random_state=42)
modelo_base.fit(X_train, y_train)

acc_base = accuracy_score(y_test, modelo_base.predict(X_test))
cv_base = cross_val_score(modelo_base, X_base, y, cv=3).mean()

print("📊 MODELO BASELINE — Solo features numéricas (precio + días de envío)")
print(f"  Accuracy en test:       {acc_base:.2%}")
print(f"  CV Score (3 folds):     {cv_base:.2%}  ← esta es la métrica principal")
print(f"\n  Nota: {acc_base:.0%} significa que el modelo acierta en {acc_base:.0%} de los casos")
print(f"  usando SOLO el precio y los días de envío — ignora completamente la reseña")

## PASO 5 (parte B) — Enriquecimiento Semántico con LLM

### Las features semánticas que generará el LLM

Estas features **no existen en el dataset original** — el LLM las infiere del texto:

| Feature generada | Tipo | Qué mide |
|-----------------|------|----------|
| `score_sentimiento` | float [-1, 1] | Sentimiento general de la reseña |
| `menciona_devolucion` | bool | Si el cliente pide devolución |
| `menciona_calidad` | bool | Si habla de la calidad del producto |
| `menciona_envio` | bool | Si menciona el envío (positivo o negativo) |
| `menciona_precio` | bool | Si habla del precio/relación calidad-precio |
| `intencion_recompra` | float [0, 1] | ¿Volvería a comprar según el texto? |
| `nivel_frustracion` | float [0, 1] | Nivel de frustración detectado |
| `longitud_percibida` | cat | Si la reseña es corta/media/larga |

### El valor de negocio de estas features

`intencion_recompra` extrae directamente la intención del cliente expresada en el texto.
Si alguien escribe "definitivamente volvería a comprar", ese `intencion_recompra=0.95` 
es una señal directa que ningún modelo de ML tabular clásico podría capturar.

In [ ]:
prompt_features = ChatPromptTemplate.from_template("""
Analiza la siguiente reseña de e-commerce y extrae features semánticas.
Responde ÚNICAMENTE con un JSON válido, sin markdown ni explicaciones.

Reseña: "{resena}"

Extrae:
{{
  "score_sentimiento": número entre -1.0 (muy negativo) y 1.0 (muy positivo),
  "menciona_devolucion": true o false,
  "menciona_calidad": true o false,
  "menciona_envio": true o false,
  "menciona_precio": true o false,
  "intencion_recompra": número entre 0.0 (nunca volvería) y 1.0 (definitivamente volvería),
  "nivel_frustracion": número entre 0.0 y 1.0,
  "longitud_percibida": "corta" o "media" o "larga"
}}
""")

chain_features = prompt_features | llm | StrOutputParser()


def extraer_features_semanticas(resena: str) -> dict:
    """Extrae features numéricas semánticas de una reseña de texto."""
    raw = chain_features.invoke({"resena": resena})
    raw = raw.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(raw)


print("Extrayendo features semánticas de todas las reseñas...")
print("(cada llamada tarda ~1-2 segundos por la API)\n")

features_semanticas = []
for i, resena in enumerate(df["resena"]):
    print(f"  [{i+1:02d}/{len(df)}] {resena[:55]}...", end=" ")
    feats = extraer_features_semanticas(resena)
    features_semanticas.append(feats)
    print(f"sent: {feats['score_sentimiento']:+.2f}  recompra: {feats['intencion_recompra']:.2f} ✅")

df_feats = pd.DataFrame(features_semanticas)
print(f"\n✅ Features semánticas extraídas:")
df_feats.head()

## PASO 5 (parte C) — Modelo Enriquecido y Comparación

### ¿Cómo se combinan las features?

```python
df_enriquecido = concat([
    df[["precio", "dias_envio"]],   # features originales (numéricas)
    df_feats[features_semanticas]   # features nuevas del LLM
], axis=1)
```

`axis=1` significa "pegar columnas" (no filas). El resultado es un DataFrame con todas las 
features juntas, listo para entrenar un modelo ML.

### Conversión de booleanos a enteros

Los valores `True/False` del LLM se convierten a `1/0` para que el GradientBoosting los procese.
El modelo ML solo habla números, no Python booleans.

In [ ]:
# Encodear la columna categórica (longitud_percibida)
le = LabelEncoder()
df_feats["longitud_enc"] = le.fit_transform(df_feats["longitud_percibida"])

# Combinar features originales + features semánticas del LLM
df_enriched = pd.concat([
    df[["precio", "dias_envio"]].reset_index(drop=True),
    df_feats.drop("longitud_percibida", axis=1).reset_index(drop=True)
], axis=1)

# Convertir booleanos a enteros (requisito del GradientBoosting)
bool_cols = ["menciona_devolucion", "menciona_calidad", "menciona_envio", "menciona_precio"]
for col in bool_cols:
    df_enriched[col] = df_enriched[col].astype(int)

print(f"Features del modelo ENRIQUECIDO ({len(df_enriched.columns)} en total):")
for col in df_enriched.columns:
    origen = "📊 original" if col in ["precio", "dias_envio"] else "🤖 LLM"
    print(f"  {origen}  {col}")

# Entrenar el modelo enriquecido
X_enrich = df_enriched
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(X_enrich, y, test_size=0.3, random_state=42)

modelo_enrich = GradientBoostingClassifier(n_estimators=50, random_state=42)
modelo_enrich.fit(X_train_e, y_train_e)

acc_enrich = accuracy_score(y_test_e, modelo_enrich.predict(X_test_e))
cv_enrich = cross_val_score(modelo_enrich, X_enrich, y, cv=3).mean()

print(f"\n{'='*55}")
print(f"📈 COMPARACIÓN DE MODELOS")
print(f"{'='*55}")
print(f"  {'Modelo':<40} {'CV Score':>10}")
print(f"  {'-'*50}")
print(f"  {'Baseline (precio + dias_envio)':<40} {cv_base:.2%}")
print(f"  {'Enriquecido (original + LLM features)':<40} {cv_enrich:.2%}")
print(f"  {'Mejora':<40} {(cv_enrich - cv_base):+.2%}")
print(f"{'='*55}")

# Feature importance
importances = pd.DataFrame({
    "feature": df_enriched.columns,
    "importance": modelo_enrich.feature_importances_
}).sort_values("importance", ascending=False)

print(f"\n🔑 IMPORTANCIA DE FEATURES (modelo enriquecido):")
print(f"   Las features del LLM más importantes para predecir recompra:\n")
for _, row in importances.iterrows():
    bar = "█" * int(row["importance"] * 35)
    origen = "LLM" if row["feature"] not in ["precio", "dias_envio"] else "NUM"
    print(f"  [{origen}] {row['feature']:<30} {bar} {row['importance']:.3f}")

## Conclusiones del Notebook T4.3

### Lo que has aprendido

1. **PATRÓN C** — LLM como generador de features:  
   `Texto → LLM → features semánticas numéricas → concat con features originales → ML mejorado`

2. **El LLM genera features con significado de negocio** que un Data Scientist humano diseñaría, 
   pero de forma automática: `intencion_recompra`, `nivel_frustracion`, `menciona_devolucion`.

3. **El baseline es obligatorio** (guía metodológica PASO 6): sin él no puedes justificar 
   el costo del LLM. Si el baseline ya da 95% de accuracy, el LLM no añade valor.

4. **Feature importance** revela qué información del texto realmente predice el target.
   En este caso, `intencion_recompra` y `score_sentimiento` suelen dominar.

5. **Concatenación de features** con `pd.concat(axis=1)` — el operador de fusión entre 
   el mundo numérico y el mundo semántico.

---

### ¿Cuándo es rentable este patrón?

- Cuando tienes columnas de texto con información de negocio valiosa (reseñas, comentarios, notas)
- Cuando el baseline sin texto tiene accuracy baja o mediocre
- Cuando las features semánticas son interpretables y validables por el negocio
- Cuando el volumen de datos permite amortizar el costo de la API del LLM

---

### Patrón clave (para memorizar)

```
Texto → LLM → features semánticas → pd.concat(features_orig, features_llm) → modelo mejorado
```